In [1]:
from wp6_data.red.db import MySQLConnection
from wp6_data.red import deps

In [2]:
async def load_data():
    db = MySQLConnection(
        host=deps.DB_HOST,
        port=deps.DB_PORT,
        user=deps.DB_USER,
        password=deps.DB_PASSWORD,
        database=deps.DB_NAME,
    )

    await db.connect()
    try:
        df_temp = await db.get_readings_by_measurement("temp")
        df_par = await db.get_readings_by_measurement("par")
        sensors = await db.get_available_sensors()
        return df_temp, df_par, sensors
    finally:
        await db.close()

df_temp, df_par, sensors = await load_data()

In [3]:
df_temp.tail(10)

,device,sensor,time,value
213133,s2101:s2101-01-temp-hum,temp,2026-04-28 14:34:53+00:00,32.97
55425,lht65_e5:lht65n-e5,temp,2026-04-28 14:35:14+00:00,29.82
184670,s1000:s1000-wetherstation,temp,2026-04-28 14:38:31+00:00,19.31
134121,lht65_ne117:lht65s-ne117-03,temp,2026-04-28 14:41:22+00:00,28.84
241734,s2103:s2103-01-temp-hum-co2,temp,2026-04-28 14:41:36+00:00,34.38
134122,lht65_ne117:lht65s-ne117,temp,2026-04-28 14:42:18+00:00,21.03
134123,lht65_ne117:lht65s-ne117-02,temp,2026-04-28 14:42:32+00:00,37.37
184671,s1000:s1000-wetherstation,temp,2026-04-28 14:43:33+00:00,19.44
55426,lht65_e5:lht65s-e5-02,temp,2026-04-28 14:44:15+00:00,26.36
213134,s2101:s2101-01-temp-hum,temp,2026-04-28 14:44:46+00:00,32.16


In [4]:
df_par.tail(10)

,device,sensor,time,value
68543,s2100:s2100-01-par,par,2026-04-28 14:29:34+00:00,1204.0
68544,s2100:s2100-10-par,par,2026-04-28 14:30:30+00:00,629.0
68545,s2100:s2100-13-par,par,2026-04-28 14:30:30+00:00,20.0
68546,s2100:s2100-11-par,par,2026-04-28 14:31:52+00:00,1077.0
68547,s2100:s2100-02-par,par,2026-04-28 14:35:23+00:00,224.0
68548,s2100:s2100-14-par,par,2026-04-28 14:36:17+00:00,94.0
68549,s2100:s2100-01-par,par,2026-04-28 14:39:32+00:00,294.0
68550,s2100:s2100-10-par,par,2026-04-28 14:40:28+00:00,709.0
68551,s2100:s2100-13-par,par,2026-04-28 14:40:38+00:00,9.0
68552,s2100:s2100-11-par,par,2026-04-28 14:41:50+00:00,649.0


In [5]:
df_par01_filtered = df_par[
    (df_par["device"] == "s2100:s2100-01-par") &
    (df_par["sensor"] == "par")
].sort_values("time")

In [8]:
df_par01_filtered.tail(10)

,device,sensor,time,value
68495,s2100:s2100-01-par,par,2026-04-28 13:09:26+00:00,1588.0
68501,s2100:s2100-01-par,par,2026-04-28 13:19:32+00:00,1405.0
68507,s2100:s2100-01-par,par,2026-04-28 13:29:28+00:00,1655.0
68513,s2100:s2100-01-par,par,2026-04-28 13:39:36+00:00,1767.0
68519,s2100:s2100-01-par,par,2026-04-28 13:49:36+00:00,1736.0
68525,s2100:s2100-01-par,par,2026-04-28 13:59:32+00:00,1642.0
68531,s2100:s2100-01-par,par,2026-04-28 14:09:32+00:00,1564.0
68537,s2100:s2100-01-par,par,2026-04-28 14:19:36+00:00,1473.0
68543,s2100:s2100-01-par,par,2026-04-28 14:29:34+00:00,1204.0
68549,s2100:s2100-01-par,par,2026-04-28 14:39:32+00:00,294.0


In [6]:
sensors

[{'table': 'lht65_ne117',
  'devices': 3,
  'readings': 78697,
  'measurements': ['temp', 'hum', 'temp_ext']},
 {'table': 's2100', 'devices': 7, 'readings': 68559, 'measurements': ['par']},
 {'table': 'dendro',
  'devices': 2,
  'readings': 55591,
  'measurements': ['adc_ch1', 'adc_ch2', 'adc_ch3']},
 {'table': 'lht65_e5',
  'devices': 2,
  'readings': 55427,
  'measurements': ['temp', 'hum', 'lux']},
 {'table': 's1000',
  'devices': 1,
  'readings': 52525,
  'measurements': ['temp',
   'hum',
   'air_press',
   'lux',
   'wind_sp',
   'wind_dir',
   'rainfall',
   'pm25',
   'pm10',
   'co2']},
 {'table': 's2107',
  'devices': 1,
  'readings': 29361,
  'measurements': ['temp_ext']},
 {'table': 's2103',
  'devices': 1,
  'readings': 28602,
  'measurements': ['temp', 'hum', 'co2']},
 {'table': 's2101',
  'devices': 1,
  'readings': 28464,
  'measurements': ['temp', 'hum']},
 {'table': 's31_lb',
  'devices': 1,
  'readings': 24449,
  'measurements': ['temp_ext', 'hum_ext']}]

In [ ]:
# write

from pathlib import Path

DATA_DIR = Path("../data") 
DATA_DIR.mkdir(parents=True, exist_ok=True)

file_path = DATA_DIR / "red_par_sample.parquet"

df_par.to_parquet(file_path, index=False)

print(f"Saved to: {file_path}")

Saved to: ..\data\red_par_sample.parquet


In [ ]:
# read

import pandas as pd

DATA_DIR = Path("../data")
file_path = DATA_DIR / "red_par_sample.parquet"

df = pd.read_parquet(file_path)

df["time"] = pd.to_datetime(df["time"], utc=True)

df.tail()

,device,sensor,time,value
0,s2100:s2100-01-par,par,2025-10-09 06:55:03+00:00,0.0
1,s2100:s2100-02-par,par,2025-10-09 07:01:07+00:00,0.0
2,s2100:s2100-01-par,par,2025-10-09 07:04:55+00:00,0.0
3,s2100:s2100-02-par,par,2025-10-09 07:11:07+00:00,0.0
4,s2100:s2100-01-par,par,2025-10-09 07:14:57+00:00,0.0
